In [20]:
import pandas as pd
import numpy as np

from pathlib import Path

In [21]:
DATA_PATH = Path("../data/processed/cleaned_data.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,1200,2.0,1.0,51.00


In [22]:
df["bhk"] = df["size"].apply(lambda x: int(str(x).split(" ")[0]))

df[["size", "bhk"]].head()

,size,bhk
0,2 BHK,2
1,4 Bedroom,4
2,3 BHK,3
3,3 BHK,3
4,2 BHK,2


In [23]:
def convert_sqft_to_number(x):

    tokens = str(x).split("-")

    if len(tokens) == 2:

        return (
            float(tokens[0]) +
            float(tokens[1])
        ) / 2

    try:

        return float(x)

    except:

        return None

In [24]:
df["total_sqft"] = df["total_sqft"].apply(convert_sqft_to_number)

In [25]:
df = df[df["total_sqft"].notnull()]

In [26]:
df[["total_sqft"]].head()

,total_sqft
0,1056.0
1,2600.0
2,1440.0
3,1521.0
4,1200.0


In [27]:
df["price_per_sqft"] = (
    df["price"] * 100000
) / df["total_sqft"]

In [28]:
def remove_pps_outliers(df):
    df_out = pd.DataFrame()

    for key, subdf in df.groupby("location"):

        mean = np.mean(subdf.price_per_sqft)
        std = np.std(subdf.price_per_sqft)

        reduced_df = subdf[
            (subdf.price_per_sqft >= (mean - std)) &
            (subdf.price_per_sqft <= (mean + std))
        ]

        df_out = pd.concat([df_out, reduced_df], ignore_index=True)

    return df_out

In [29]:
def remove_bhk_outliers(df):

    exclude_indices = np.array([])

    for location, location_df in df.groupby("location"):

        bhk_stats = {}

        for bhk, bhk_df in location_df.groupby("bhk"):

            bhk_stats[bhk] = {
                "mean": np.mean(bhk_df.price_per_sqft),
                "count": bhk_df.shape[0]
            }

        for bhk, bhk_df in location_df.groupby("bhk"):

            stats = bhk_stats.get(bhk - 1)

            if stats and stats["count"] > 5:

                exclude_indices = np.append(
                    exclude_indices,
                    bhk_df[
                        bhk_df.price_per_sqft < stats["mean"]
                    ].index.values
                )

    return df.drop(exclude_indices.astype(int))

In [30]:
df = remove_pps_outliers(df)

In [31]:
df = remove_bhk_outliers(df)

print(df.shape)

(8451, 10)


In [32]:
# Drop unnecessary columns
df.drop(columns=["availability"], inplace=True)

df.drop(columns=["price_per_sqft"], inplace=True)

df.head()

,area_type,location,size,total_sqft,bath,balcony,price,bhk
0,Plot Area,1 Annasandrapalya,11 Bedroom,1200.0,6.0,3.0,150.0,11
1,Super built-up Area,1 Giri Nagar,11 BHK,5000.0,9.0,3.0,360.0,11
2,Super built-up Area,1 Immadihalli,11 BHK,6000.0,12.0,2.0,150.0,11
3,Plot Area,1 Ramamurthy Nagar,11 Bedroom,1200.0,11.0,0.0,170.0,11
4,Super built-up Area,12th cross srinivas nagar banshankari 3rd stage,1 BHK,1800.0,1.0,1.0,200.0,1


In [33]:
df = df[
    df["total_sqft"] / df["bhk"] >= 300
]

In [34]:
location_stats = df["location"].value_counts()

location_stats.head()

location
Whitefield               291
Sarjapur  Road           249
Electronic City          159
Raja Rajeshwari Nagar    129
Marathahalli             113
Name: count, dtype: int64

In [35]:
location_stats_less_than_10 = location_stats[
    location_stats <= 10
]

location_stats_less_than_10

location
Basaveshwara Nagar      10
Billekahalli            10
Chikka Tirupathi        10
Cunningham Road         10
Dommasandra             10
                        ..
sapthagiri Layout        1
sarjapura main road      1
singapura paradise       1
white field,kadugodi     1
whitefiled               1
Name: count, Length: 1022, dtype: int64

In [36]:
df["location"] = df["location"].apply(

    lambda x:

    "Other"

    if x in location_stats_less_than_10

    else x

)

In [37]:
df["location"].value_counts()

location
Other                    2403
Whitefield                291
Sarjapur  Road            249
Electronic City           159
Raja Rajeshwari Nagar     129
                         ... 
Nagavara                   11
Poorna Pragna Layout       11
Sector 7 HSR Layout        11
Sultan Palaya              11
Ulsoor                     11
Name: count, Length: 172, dtype: int64

In [38]:
df.drop(
    columns=[
        "size"
    ],
    inplace=True
)

In [39]:
output_dir = Path("../data/processed")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    output_dir / "featured_data.csv",
    index=False
)